In [ ]:
# Install required libraries
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 62.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Hugging Face Authentication
# ============================================================

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Please add your Hugging Face token "
        "in Google Colab Secrets."
    )

login(token=HF_TOKEN)


In [ ]:
# ============================================================
# Model Configuration
# ============================================================

# ONLY CHANGE THIS MODEL NAME
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"



In [ ]:
# ============================================================
# Device Configuration
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Model:", MODEL_ID)


Device: cuda
Model: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# ============================================================
# Load Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# ============================================================
# Load Model
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    token=HF_TOKEN
)

model.eval()

print("Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def chat(
    prompt,
    system_prompt="You are a helpful AI assistant.",
    max_new_tokens=200,
    temperature=0.7
):
    """
    Generate a response from a Hugging Face
    text-to-text/chat model.
    """

    # Create chat messages
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Convert messages into the model's chat format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the formatted text
    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    # Move input tensors to the model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the original prompt tokens
    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    # Decode generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
response = chat(
    "What is Artificial Intelligence? Explain it in simple words."
)

print("AI:", response)

AI: Artificial Intelligence (AI) refers to the ability of machines or computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, and problem-solving.

In simpler terms, AI involves creating intelligent software programs that can learn from data, make decisions based on patterns they identify, and adapt their behavior over time. These programs use algorithms and machine learning techniques to analyze large amounts of information and draw insights, enabling them to perform complex tasks without being explicitly programmed for each specific task.

Some examples of AI applications include chatbots, virtual assistants like Siri or Alexa, self-driving cars, recommendation engines, image recognition, speech recognition, fraud detection, and medical diagnosis. AI technology continues to evolve rapidly, with new advancements constantly emerging, making it an exciting field with significant potential to transform various industries and improve our d

In [ ]:
response = chat(
    "What is the difference between Machine Learning and Deep Learning?"
)

print("AI:", response)

AI: Machine learning (ML) and deep learning (DL) are two related but distinct fields of artificial intelligence that focus on teaching computers to learn from data.

Machine learning involves using algorithms to analyze patterns in data and make predictions or decisions based on those patterns. This can be done through supervised, unsupervised, or semi-supervised learning methods, where the algorithm learns by being provided with labeled or unlabeled examples of input-output pairs.

Deep learning, on the other hand, is a subset of machine learning that uses neural networks with multiple layers to automatically extract features from raw data. These networks are trained on large amounts of labeled data to identify complex patterns and relationships within the data. Deep learning has achieved impressive results in areas such as image recognition, speech recognition, and natural language processing.

In summary, while both ML and DL involve training algorithms on data, deep learning specif